In [2]:
# Data Lab - Extract transferred Case Study 2 files

from pathlib import Path
import zipfile

home = Path("/home/jovyan")

zip_path = home / "Case_Study_2_Datalab_Transfer.zip"

project_path = home / "Case_Study_2_Medical_Consultation_AI"

print("DATA LAB PROJECT EXTRACTION")
print("=" * 60)

print("ZIP exists:", zip_path.exists())

# Create project folder
project_path.mkdir(
    parents=True,
    exist_ok=True
)

# Extract ZIP
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(project_path)

print("Extraction completed:", project_path.exists())

# Important paths
audio_folder = (
    project_path /
    "data/pilot_mixed_audio"
)

reference_folder = (
    project_path /
    "data/reference_transcripts"
)

pilot_manifest = (
    project_path /
    "data/primock57_pilot_evaluation_set.csv"
)

whisper_results = (
    project_path /
    "results/asr/whisper_large_v3"
)

# Count files
audio_files = list(
    audio_folder.glob("*.wav")
) if audio_folder.exists() else []

reference_files = list(
    reference_folder.iterdir()
) if reference_folder.exists() else []

print("\n" + "=" * 60)
print("TRANSFER VERIFICATION")
print("=" * 60)

print("Project folder:", project_path)
print("Pilot audio folder exists:", audio_folder.exists())
print("Pilot WAV files:", len(audio_files))

print(
    "Reference transcript folder exists:",
    reference_folder.exists()
)

print(
    "Reference transcript files:",
    len(reference_files)
)

print(
    "Pilot manifest exists:",
    pilot_manifest.exists()
)

print(
    "Existing Whisper result folder:",
    whisper_results.exists()
)

print("\nPilot audio files:")

for file in sorted(audio_files):
    print("✓", file.name)

print("\n" + "=" * 60)

if (
    len(audio_files) == 10
    and pilot_manifest.exists()
    and whisper_results.exists()
):
    print("TRANSFER STATUS: SUCCESS")
else:
    print("TRANSFER STATUS: CHECK REQUIRED")

DATA LAB PROJECT EXTRACTION
ZIP exists: True
Extraction completed: True

TRANSFER VERIFICATION
Project folder: /home/jovyan/Case_Study_2_Medical_Consultation_AI
Pilot audio folder exists: True
Pilot WAV files: 10
Reference transcript folder exists: True
Reference transcript files: 40
Pilot manifest exists: True
Existing Whisper result folder: True

Pilot audio files:
✓ day1_consultation01_mixed.wav
✓ day1_consultation07_mixed.wav
✓ day2_consultation01_mixed.wav
✓ day2_consultation05_mixed.wav
✓ day3_consultation06_mixed.wav
✓ day3_consultation09_mixed.wav
✓ day4_consultation03_mixed.wav
✓ day4_consultation10_mixed.wav
✓ day5_consultation03_mixed.wav
✓ day5_consultation12_mixed.wav

TRANSFER STATUS: SUCCESS


In [1]:
# Install reproducible ASR evaluation packages

%pip install -q faster-whisper==1.2.1 jiwer==4.0.0

import faster_whisper
import jiwer

print("ASR PACKAGE CHECK")
print("=" * 50)
print("faster-whisper version:", faster_whisper.__version__)
print("jiwer version:", jiwer.__version__ if hasattr(jiwer, "__version__") else "4.0.0")
print("\nPACKAGE STATUS: READY")

Note: you may need to restart the kernel to use updated packages.
ASR PACKAGE CHECK
faster-whisper version: 1.2.1
jiwer version: 4.0.0

PACKAGE STATUS: READY


In [3]:
# Whisper Large-v3 - H200 compatibility test
# Uses the same settings as the original Colab baseline

from pathlib import Path
from faster_whisper import WhisperModel
import torch
import time
import json

project_path = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

audio_path = (
    project_path /
    "data/pilot_mixed_audio/day1_consultation01_mixed.wav"
)

output_folder = (
    project_path /
    "results/asr/whisper_large_v3/datalab_compatibility_test"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("H200 COMPATIBILITY TEST")
print("=" * 60)

print("Audio exists:", audio_path.exists())
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

if not audio_path.exists():
    raise FileNotFoundError(
        f"Audio file not found: {audio_path}"
    )

# ---------------------------------------------------------
# Load Whisper Large-v3
# ---------------------------------------------------------

print("\nLoading Whisper Large-v3...")

load_start = time.perf_counter()

model = WhisperModel(
    "large-v3",
    device="cuda",
    compute_type="float16"
)

model_load_seconds = (
    time.perf_counter() - load_start
)

print(
    "Model load time:",
    round(model_load_seconds, 2),
    "seconds"
)

# ---------------------------------------------------------
# Transcribe
# ---------------------------------------------------------

print("\nTranscribing consultation...")

start_time = time.perf_counter()

segments_generator, info = model.transcribe(
    str(audio_path),
    language="en",
    beam_size=5,
    vad_filter=True,
    word_timestamps=True,
    condition_on_previous_text=True
)

segments = []

for segment in segments_generator:

    words = []

    if segment.words:
        for word in segment.words:
            words.append({
                "start":
                    round(word.start, 3)
                    if word.start is not None
                    else None,

                "end":
                    round(word.end, 3)
                    if word.end is not None
                    else None,

                "word": word.word,

                "probability":
                    round(word.probability, 4)
                    if word.probability is not None
                    else None
            })

    segments.append({
        "id": segment.id,
        "start": round(segment.start, 3),
        "end": round(segment.end, 3),
        "text": segment.text.strip(),
        "words": words
    })

transcription_seconds = (
    time.perf_counter() - start_time
)

full_transcript = " ".join(
    segment["text"]
    for segment in segments
).strip()

audio_duration_seconds = float(info.duration)

real_time_factor = (
    transcription_seconds /
    audio_duration_seconds
)

# ---------------------------------------------------------
# Save Data Lab compatibility result
# ---------------------------------------------------------

transcript_path = (
    output_folder /
    "day1_consultation01_datalab_transcript.txt"
)

segments_path = (
    output_folder /
    "day1_consultation01_datalab_segments.json"
)

metadata_path = (
    output_folder /
    "day1_consultation01_datalab_metadata.json"
)

transcript_path.write_text(
    full_transcript,
    encoding="utf-8"
)

with open(
    segments_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        segments,
        file,
        indent=2,
        ensure_ascii=False
    )

metadata = {
    "consultation_id": "day1_consultation01",
    "model": "large-v3",
    "implementation": "faster-whisper",
    "environment": "SRH Data Lab",
    "gpu": torch.cuda.get_device_name(0),
    "compute_type": "float16",
    "audio_duration_seconds":
        round(audio_duration_seconds, 2),
    "model_load_seconds":
        round(model_load_seconds, 2),
    "transcription_seconds":
        round(transcription_seconds, 2),
    "real_time_factor":
        round(real_time_factor, 4),
    "number_of_segments":
        len(segments),
    "transcript_word_count":
        len(full_transcript.split()),
    "language":
        info.language,
    "language_probability":
        round(info.language_probability, 4)
}

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        metadata,
        file,
        indent=2
    )

# ---------------------------------------------------------
# Final summary
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("H200 COMPATIBILITY TEST RESULT")
print("=" * 60)

print(
    "Audio duration:",
    round(audio_duration_seconds, 2),
    "seconds"
)

print(
    "Model load time:",
    round(model_load_seconds, 2),
    "seconds"
)

print(
    "Transcription time:",
    round(transcription_seconds, 2),
    "seconds"
)

print(
    "Real-time factor:",
    round(real_time_factor, 4)
)

print(
    "Segments:",
    len(segments)
)

print(
    "Transcript words:",
    len(full_transcript.split())
)

print(
    "Language:",
    info.language
)

print(
    "Language probability:",
    round(info.language_probability, 4)
)

print("\nTranscript preview:")
print(full_transcript[:1000])

print("\nSaved to:")
print(output_folder)

H200 COMPATIBILITY TEST
Audio exists: True
CUDA available: True
GPU: NVIDIA H200 NVL
GPU memory: 139.8 GB

Loading Whisper Large-v3...


vocabulary.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Model load time: 30.45 seconds

Transcribing consultation...

H200 COMPATIBILITY TEST RESULT
Audio duration: 457.92 seconds
Model load time: 30.45 seconds
Transcription time: 24.77 seconds
Real-time factor: 0.0541
Segments: 137
Transcript words: 1290
Language: en
Language probability: 1

Transcript preview:
Hello. Hello, how are you? Hi. Shall we stop? Yeah, okay. Hello, good morning sir, how can I help you this morning? Hi, I've just had some diarrhoea for the last three days and it's been affecting me. I need to stay close to the toilet and yeah, it's been affecting my day-to-day activities. Yeah, I'm sorry to hear that. And when you say diarrhoea, what do you mean by diarrhoea? Do you mean you're going to the toilet more often or are your stools more loose? Yeah, so it's like loose and watery stool, going to the toilet quite often and like some pain in my like lower stomach. Okay, and how many times a day are you going, let's say, for the last couple of days? Probably like six, seve

In [4]:
# Compare H200 Whisper transcript against the same reference transcript

from pathlib import Path
from jiwer import process_words
import unicodedata
import json
import re

project_path = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

# H200 transcript
hypothesis_path = (
    project_path /
    "results/asr/whisper_large_v3/"
    "datalab_compatibility_test/"
    "day1_consultation01_datalab_transcript.txt"
)

# Find the correct reference transcript automatically
reference_folder = (
    project_path /
    "data/reference_transcripts"
)

candidates = list(
    reference_folder.glob(
        "day1_consultation01*reference_plain.txt"
    )
)

print("H200 WER VERIFICATION")
print("=" * 60)

print("Hypothesis exists:", hypothesis_path.exists())
print("Reference candidates found:", len(candidates))

if not candidates:
    raise FileNotFoundError(
        "Reference plain transcript not found."
    )

reference_path = candidates[0]

print("Reference file:", reference_path.name)

reference_text = reference_path.read_text(
    encoding="utf-8"
)

hypothesis_text = hypothesis_path.read_text(
    encoding="utf-8"
)

# Same normalization approach used previously
def normalize_text(text):

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = text.lower()

    cleaned = []

    for char in text:

        category = unicodedata.category(char)

        # Replace punctuation/symbols with spaces
        if category.startswith("P") or category.startswith("S"):
            cleaned.append(" ")
        else:
            cleaned.append(char)

    text = "".join(cleaned)

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


reference_normalized = normalize_text(
    reference_text
)

hypothesis_normalized = normalize_text(
    hypothesis_text
)

result = process_words(
    reference_normalized,
    hypothesis_normalized
)

reference_words = len(
    reference_normalized.split()
)

hypothesis_words = len(
    hypothesis_normalized.split()
)

total_errors = (
    result.substitutions +
    result.deletions +
    result.insertions
)

metrics = {
    "environment": "SRH Data Lab H200",
    "model": "Whisper Large-v3",
    "reference_words": reference_words,
    "hypothesis_words": hypothesis_words,
    "wer": round(result.wer, 4),
    "wer_percent": round(result.wer * 100, 2),
    "mer": round(result.mer, 4),
    "wil": round(result.wil, 4),
    "hits": result.hits,
    "substitutions": result.substitutions,
    "deletions": result.deletions,
    "insertions": result.insertions,
    "total_errors": total_errors
}

output_path = (
    project_path /
    "results/asr/whisper_large_v3/"
    "datalab_compatibility_test/"
    "day1_consultation01_datalab_wer_metrics.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        metrics,
        file,
        indent=2
    )

print("\n" + "=" * 60)
print("H200 WER RESULT")
print("=" * 60)

print("Reference words:", reference_words)
print("Hypothesis words:", hypothesis_words)
print("WER:", round(result.wer * 100, 2), "%")
print("MER:", round(result.mer, 4))
print("WIL:", round(result.wil, 4))
print("Hits:", result.hits)
print("Substitutions:", result.substitutions)
print("Deletions:", result.deletions)
print("Insertions:", result.insertions)
print("Total errors:", total_errors)

print("\nPrevious Colab T4 baseline:")
print("WER: 21.4 %")
print("Substitutions: 79")
print("Deletions: 189")
print("Insertions: 46")
print("Total errors: 314")

print("\nSaved:")
print(output_path)

H200 WER VERIFICATION
Hypothesis exists: True
Reference candidates found: 1
Reference file: day1_consultation01_reference_plain.txt

H200 WER RESULT
Reference words: 1467
Hypothesis words: 1344
WER: 21.47 %
MER: 0.2062
WIL: 0.2537
Hits: 1213
Substitutions: 70
Deletions: 184
Insertions: 61
Total errors: 315

Previous Colab T4 baseline:
WER: 21.4 %
Substitutions: 79
Deletions: 189
Insertions: 46
Total errors: 314

Saved:
/home/jovyan/Case_Study_2_Medical_Consultation_AI/results/asr/whisper_large_v3/datalab_compatibility_test/day1_consultation01_datalab_wer_metrics.json


In [5]:
# Task 5E - Transcribe all 10 pilot consultations on Data Lab H200

from pathlib import Path
from faster_whisper import WhisperModel
import pandas as pd
import torch
import time
import json

project_path = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

pilot_manifest_path = (
    project_path /
    "data/primock57_pilot_evaluation_set.csv"
)

audio_folder = (
    project_path /
    "data/pilot_mixed_audio"
)

output_folder = (
    project_path /
    "results/asr/whisper_large_v3/datalab_pilot"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# Load pilot manifest
# ---------------------------------------------------------

pilot_df = pd.read_csv(
    pilot_manifest_path
)

print("WHISPER LARGE-V3 - H200 PILOT TRANSCRIPTION")
print("=" * 65)

print("Pilot manifest exists:", pilot_manifest_path.exists())
print("Pilot consultations:", len(pilot_df))
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# ---------------------------------------------------------
# Load Whisper
# ---------------------------------------------------------

print("\nLoading Whisper Large-v3...")

load_start = time.perf_counter()

model = WhisperModel(
    "large-v3",
    device="cuda",
    compute_type="float16"
)

model_load_seconds = (
    time.perf_counter() - load_start
)

print(
    "Model loaded in:",
    round(model_load_seconds, 2),
    "seconds"
)

# ---------------------------------------------------------
# Transcribe each consultation
# ---------------------------------------------------------

summary_rows = []

overall_start = time.perf_counter()

print("\nSTARTING PILOT TRANSCRIPTION")
print("=" * 65)

for index, row in pilot_df.iterrows():

    consultation_id = row["consultation_id"]

    audio_path = (
        audio_folder /
        f"{consultation_id}_mixed.wav"
    )

    transcript_path = (
        output_folder /
        f"{consultation_id}_transcript.txt"
    )

    segments_path = (
        output_folder /
        f"{consultation_id}_segments.json"
    )

    metadata_path = (
        output_folder /
        f"{consultation_id}_metadata.json"
    )

    print(
        f"\n[{index + 1}/{len(pilot_df)}] "
        f"{consultation_id}"
    )

    if not audio_path.exists():

        print("✗ Audio missing")

        summary_rows.append({
            "consultation_id": consultation_id,
            "status": "audio_missing"
        })

        continue

    start_time = time.perf_counter()

    segments_generator, info = model.transcribe(
        str(audio_path),
        language="en",
        beam_size=5,
        vad_filter=True,
        word_timestamps=True,
        condition_on_previous_text=True
    )

    segments = []

    for segment in segments_generator:

        words = []

        if segment.words:

            for word in segment.words:

                words.append({
                    "start":
                        round(word.start, 3)
                        if word.start is not None
                        else None,

                    "end":
                        round(word.end, 3)
                        if word.end is not None
                        else None,

                    "word": word.word,

                    "probability":
                        round(word.probability, 4)
                        if word.probability is not None
                        else None
                })

        segments.append({
            "id": segment.id,
            "start": round(segment.start, 3),
            "end": round(segment.end, 3),
            "text": segment.text.strip(),
            "words": words
        })

    transcription_seconds = (
        time.perf_counter() - start_time
    )

    full_transcript = " ".join(
        segment["text"]
        for segment in segments
    ).strip()

    audio_duration_seconds = float(
        info.duration
    )

    real_time_factor = (
        transcription_seconds /
        audio_duration_seconds
    )

    # -----------------------------------------------------
    # Save transcript
    # -----------------------------------------------------

    transcript_path.write_text(
        full_transcript,
        encoding="utf-8"
    )

    # -----------------------------------------------------
    # Save detailed segment output
    # -----------------------------------------------------

    with open(
        segments_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            segments,
            file,
            indent=2,
            ensure_ascii=False
        )

    # -----------------------------------------------------
    # Save metadata
    # -----------------------------------------------------

    metadata = {
        "consultation_id": consultation_id,
        "status": "success",
        "model": "Whisper Large-v3",
        "implementation": "faster-whisper 1.2.1",
        "environment": "SRH Data Lab",
        "gpu": torch.cuda.get_device_name(0),
        "compute_type": "float16",
        "language": info.language,
        "language_probability":
            round(info.language_probability, 4),

        "audio_duration_seconds":
            round(audio_duration_seconds, 2),

        "transcription_seconds":
            round(transcription_seconds, 2),

        "real_time_factor":
            round(real_time_factor, 4),

        "number_of_segments":
            len(segments),

        "transcript_word_count":
            len(full_transcript.split())
    }

    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            metadata,
            file,
            indent=2
        )

    summary_rows.append(metadata)

    print("✓ Completed")
    print(
        "  Audio:",
        round(audio_duration_seconds / 60, 2),
        "minutes"
    )
    print(
        "  Transcription:",
        round(transcription_seconds, 2),
        "seconds"
    )
    print(
        "  RTF:",
        round(real_time_factor, 4)
    )
    print(
        "  Words:",
        len(full_transcript.split())
    )

# ---------------------------------------------------------
# Create summary
# ---------------------------------------------------------

overall_seconds = (
    time.perf_counter() - overall_start
)

summary_df = pd.DataFrame(
    summary_rows
)

summary_path = (
    output_folder /
    "whisper_large_v3_h200_pilot_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

completed = (
    summary_df["status"]
    .eq("success")
    .sum()
)

print("\n" + "=" * 65)
print("WHISPER LARGE-V3 H200 PILOT SUMMARY")
print("=" * 65)

print(
    "Pilot consultations:",
    len(pilot_df)
)

print(
    "Successfully transcribed:",
    completed
)

print(
    "Total audio duration:",
    round(
        summary_df[
            "audio_duration_seconds"
        ].sum() / 60,
        2
    ),
    "minutes"
)

print(
    "Total transcription time:",
    round(
        summary_df[
            "transcription_seconds"
        ].sum(),
        2
    ),
    "seconds"
)

print(
    "Average RTF:",
    round(
        summary_df[
            "real_time_factor"
        ].mean(),
        4
    )
)

print(
    "Total wall-clock experiment time:",
    round(
        overall_seconds,
        2
    ),
    "seconds"
)

print("\nRESULT TABLE")
print("-" * 65)

print(
    summary_df[
        [
            "consultation_id",
            "audio_duration_seconds",
            "transcription_seconds",
            "real_time_factor",
            "transcript_word_count"
        ]
    ].to_string(index=False)
)

print("\nSummary saved:")
print(summary_path)

WHISPER LARGE-V3 - H200 PILOT TRANSCRIPTION
Pilot manifest exists: True
Pilot consultations: 10
CUDA available: True
GPU: NVIDIA H200 NVL

Loading Whisper Large-v3...
Model loaded in: 2.2 seconds

STARTING PILOT TRANSCRIPTION

[1/10] day1_consultation01
✓ Completed
  Audio: 7.63 minutes
  Transcription: 16.09 seconds
  RTF: 0.0351
  Words: 1290

[2/10] day1_consultation07
✓ Completed
  Audio: 14.3 minutes
  Transcription: 38.45 seconds
  RTF: 0.0448
  Words: 2521

[3/10] day2_consultation01
✓ Completed
  Audio: 5.49 minutes
  Transcription: 13.11 seconds
  RTF: 0.0398
  Words: 1104

[4/10] day2_consultation05
✓ Completed
  Audio: 10.6 minutes
  Transcription: 21.16 seconds
  RTF: 0.0332
  Words: 1754

[5/10] day3_consultation06
✓ Completed
  Audio: 3.81 minutes
  Transcription: 12.58 seconds
  RTF: 0.055
  Words: 605

[6/10] day3_consultation09
✓ Completed
  Audio: 11.87 minutes
  Transcription: 41.49 seconds
  RTF: 0.0582
  Words: 1951

[7/10] day4_consultation03
✓ Completed
  Audio: 

In [6]:
WHISPER LARGE-V3 - H200 PILOT TRANSCRIPTION
=================================================================
Pilot manifest exists: True
Pilot consultations: 10
CUDA available: True
GPU: NVIDIA H200 NVL

Loading Whisper Large-v3...
Model loaded in: 2.2 seconds

STARTING PILOT TRANSCRIPTION
=================================================================

[1/10] day1_consultation01
✓ Completed
  Audio: 7.63 minutes
  Transcription: 16.09 seconds
  RTF: 0.0351
  Words: 1290

[2/10] day1_consultation07
✓ Completed
  Audio: 14.3 minutes
  Transcription: 38.45 seconds
  RTF: 0.0448
  Words: 2521

[3/10] day2_consultation01
✓ Completed
  Audio: 5.49 minutes
  Transcription: 13.11 seconds
  RTF: 0.0398
  Words: 1104

[4/10] day2_consultation05
✓ Completed
  Audio: 10.6 minutes
  Transcription: 21.16 seconds
  RTF: 0.0332
  Words: 1754

[5/10] day3_consultation06
✓ Completed
  Audio: 3.81 minutes
  Transcription: 12.58 seconds
  RTF: 0.055
  Words: 605

[6/10] day3_consultation09
✓ Completed
  Audio: 11.87 minutes
  Transcription: 41.49 seconds
  RTF: 0.0582
  Words: 1951

[7/10] day4_consultation03
✓ Completed
  Audio: 6.37 minutes
  Transcription: 17.63 seconds
  RTF: 0.0461
  Words: 959

[8/10] day4_consultation10
✓ Completed
  Audio: 12.41 minutes
  Transcription: 29.8 seconds
  RTF: 0.04
  Words: 1634

[9/10] day5_consultation12
✓ Completed
  Audio: 5.43 minutes
  Transcription: 17.17 seconds
  RTF: 0.0527
  Words: 735

[10/10] day5_consultation03
✓ Completed
  Audio: 12.94 minutes
  Transcription: 39.85 seconds
  RTF: 0.0513
  Words: 2312

=================================================================
WHISPER LARGE-V3 H200 PILOT SUMMARY
=================================================================
Pilot consultations: 10
Successfully transcribed: 10
Total audio duration: 90.86 minutes
Total transcription time: 247.33 seconds
Average RTF: 0.0456
Total wall-clock experiment time: 247.42 seconds

RESULT TABLE
-----------------------------------------------------------------
    consultation_id  audio_duration_seconds  transcription_seconds  real_time_factor  transcript_word_count
day1_consultation01                  457.92                  16.09            0.0351                   1290
day1_consultation07                  858.24                  38.45            0.0448                   2521
day2_consultation01                  329.16                  13.11            0.0398                   1104
day2_consultation05                  636.30                  21.16            0.0332                   1754
day3_consultation06                  228.60                  12.58            0.0550                    605
day3_consultation09                  712.44                  41.49            0.0582                   1951
day4_consultation03                  382.32                  17.63            0.0461                    959
day4_consultation10                  744.84                  29.80            0.0400                   1634
day5_consultation12                  325.56                  17.17            0.0527                    735
day5_consultation03                  776.34                  39.85            0.0513                   2312

Summary saved:
/home/jovyan/Case_Study_2_Medical_Consultation_AI/results/asr/whisper_large_v3/datalab_pilot/whisper_large_v3_h200_pilot_summary.csv

SyntaxError: invalid character '✓' (U+2713) (502373452.py, line 15)

In [7]:
# Task 5F - WER evaluation for all 10 Whisper Large-v3 pilot transcripts

from pathlib import Path
from jiwer import process_words
import pandas as pd
import unicodedata
import re
import json

project_path = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

pilot_manifest_path = (
    project_path /
    "data/primock57_pilot_evaluation_set.csv"
)

reference_folder = (
    project_path /
    "data/reference_transcripts"
)

transcript_folder = (
    project_path /
    "results/asr/whisper_large_v3/datalab_pilot"
)

evaluation_folder = (
    transcript_folder /
    "evaluation"
)

evaluation_folder.mkdir(
    parents=True,
    exist_ok=True
)

pilot_df = pd.read_csv(
    pilot_manifest_path
)

# ---------------------------------------------------------
# Same text normalization used in earlier WER evaluation
# ---------------------------------------------------------

def normalize_text(text):

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = text.lower()

    cleaned = []

    for char in text:

        category = unicodedata.category(char)

        if (
            category.startswith("P")
            or category.startswith("S")
        ):
            cleaned.append(" ")
        else:
            cleaned.append(char)

    text = "".join(cleaned)

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


print("WHISPER LARGE-V3 - PILOT WER EVALUATION")
print("=" * 70)

results = []

for index, row in pilot_df.iterrows():

    consultation_id = row["consultation_id"]

    reference_path = (
        reference_folder /
        f"{consultation_id}_reference_plain.txt"
    )

    hypothesis_path = (
        transcript_folder /
        f"{consultation_id}_transcript.txt"
    )

    print(
        f"\n[{index + 1}/{len(pilot_df)}] "
        f"{consultation_id}"
    )

    if not reference_path.exists():

        print("✗ Reference missing")

        results.append({
            "consultation_id": consultation_id,
            "status": "reference_missing"
        })

        continue

    if not hypothesis_path.exists():

        print("✗ ASR transcript missing")

        results.append({
            "consultation_id": consultation_id,
            "status": "hypothesis_missing"
        })

        continue

    reference_text = reference_path.read_text(
        encoding="utf-8"
    )

    hypothesis_text = hypothesis_path.read_text(
        encoding="utf-8"
    )

    reference_normalized = normalize_text(
        reference_text
    )

    hypothesis_normalized = normalize_text(
        hypothesis_text
    )

    result = process_words(
        reference_normalized,
        hypothesis_normalized
    )

    reference_words = len(
        reference_normalized.split()
    )

    hypothesis_words = len(
        hypothesis_normalized.split()
    )

    total_errors = (
        result.substitutions
        + result.deletions
        + result.insertions
    )

    row_result = {
        "consultation_id": consultation_id,
        "status": "success",
        "reference_words": reference_words,
        "hypothesis_words": hypothesis_words,
        "hits": result.hits,
        "substitutions": result.substitutions,
        "deletions": result.deletions,
        "insertions": result.insertions,
        "total_errors": total_errors,
        "wer": result.wer,
        "wer_percent": result.wer * 100,
        "mer": result.mer,
        "wil": result.wil
    }

    results.append(
        row_result
    )

    print(
        "✓ WER:",
        round(result.wer * 100, 2),
        "%"
    )

    print(
        "  Ref words:",
        reference_words,
        "| Errors:",
        total_errors
    )


results_df = pd.DataFrame(
    results
)

successful_df = results_df[
    results_df["status"] == "success"
].copy()

# ---------------------------------------------------------
# Aggregate metrics
# ---------------------------------------------------------

total_reference_words = int(
    successful_df[
        "reference_words"
    ].sum()
)

total_hypothesis_words = int(
    successful_df[
        "hypothesis_words"
    ].sum()
)

total_hits = int(
    successful_df[
        "hits"
    ].sum()
)

total_substitutions = int(
    successful_df[
        "substitutions"
    ].sum()
)

total_deletions = int(
    successful_df[
        "deletions"
    ].sum()
)

total_insertions = int(
    successful_df[
        "insertions"
    ].sum()
)

total_errors = (
    total_substitutions
    + total_deletions
    + total_insertions
)

# Corpus / micro WER
corpus_wer = (
    total_errors /
    total_reference_words
)

# Macro WER = each consultation weighted equally
macro_wer = (
    successful_df[
        "wer"
    ].mean()
)

median_wer = (
    successful_df[
        "wer"
    ].median()
)

best_row = successful_df.loc[
    successful_df["wer"].idxmin()
]

worst_row = successful_df.loc[
    successful_df["wer"].idxmax()
]

# ---------------------------------------------------------
# Save detailed CSV
# ---------------------------------------------------------

csv_path = (
    evaluation_folder /
    "whisper_large_v3_h200_pilot_wer_results.csv"
)

results_df.to_csv(
    csv_path,
    index=False
)

summary = {
    "model": "Whisper Large-v3",
    "implementation": "faster-whisper 1.2.1",
    "environment": "SRH Data Lab H200",
    "consultations_evaluated": int(
        len(successful_df)
    ),
    "total_reference_words": total_reference_words,
    "total_hypothesis_words": total_hypothesis_words,
    "total_hits": total_hits,
    "total_substitutions": total_substitutions,
    "total_deletions": total_deletions,
    "total_insertions": total_insertions,
    "total_errors": total_errors,
    "corpus_wer": round(
        corpus_wer,
        4
    ),
    "corpus_wer_percent": round(
        corpus_wer * 100,
        2
    ),
    "macro_wer": round(
        macro_wer,
        4
    ),
    "macro_wer_percent": round(
        macro_wer * 100,
        2
    ),
    "median_wer_percent": round(
        median_wer * 100,
        2
    ),
    "best_consultation":
        best_row["consultation_id"],
    "best_wer_percent": round(
        best_row["wer"] * 100,
        2
    ),
    "worst_consultation":
        worst_row["consultation_id"],
    "worst_wer_percent": round(
        worst_row["wer"] * 100,
        2
    )
}

json_path = (
    evaluation_folder /
    "whisper_large_v3_h200_pilot_wer_summary.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        summary,
        file,
        indent=2
    )

# ---------------------------------------------------------
# Final report
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("WHISPER LARGE-V3 PILOT WER SUMMARY")
print("=" * 70)

print(
    "Consultations evaluated:",
    len(successful_df)
)

print(
    "Total reference words:",
    total_reference_words
)

print(
    "Total hypothesis words:",
    total_hypothesis_words
)

print(
    "\nCorpus WER:",
    round(corpus_wer * 100, 2),
    "%"
)

print(
    "Macro-average WER:",
    round(macro_wer * 100, 2),
    "%"
)

print(
    "Median WER:",
    round(median_wer * 100, 2),
    "%"
)

print("\nError totals:")

print(
    "Substitutions:",
    total_substitutions
)

print(
    "Deletions:",
    total_deletions
)

print(
    "Insertions:",
    total_insertions
)

print(
    "Total errors:",
    total_errors
)

print(
    "\nBest consultation:",
    best_row["consultation_id"],
    "-",
    round(best_row["wer"] * 100, 2),
    "%"
)

print(
    "Worst consultation:",
    worst_row["consultation_id"],
    "-",
    round(worst_row["wer"] * 100, 2),
    "%"
)

print("\nPER-CONSULTATION WER")
print("-" * 70)

print(
    successful_df[
        [
            "consultation_id",
            "reference_words",
            "hypothesis_words",
            "substitutions",
            "deletions",
            "insertions",
            "wer_percent"
        ]
    ].round({
        "wer_percent": 2
    }).to_string(index=False)
)

print("\nSaved:")
print(csv_path)
print(json_path)

WHISPER LARGE-V3 - PILOT WER EVALUATION

[1/10] day1_consultation01
✓ WER: 21.47 %
  Ref words: 1467 | Errors: 315

[2/10] day1_consultation07
✓ WER: 20.1 %
  Ref words: 2910 | Errors: 585

[3/10] day2_consultation01
✓ WER: 19.16 %
  Ref words: 1221 | Errors: 234

[4/10] day2_consultation05
✓ WER: 14.6 %
  Ref words: 1986 | Errors: 290

[5/10] day3_consultation06
✓ WER: 16.88 %
  Ref words: 616 | Errors: 104

[6/10] day3_consultation09
✓ WER: 13.95 %
  Ref words: 2200 | Errors: 307

[7/10] day4_consultation03
✓ WER: 12.35 %
  Ref words: 1069 | Errors: 132

[8/10] day4_consultation10
✓ WER: 17.58 %
  Ref words: 1786 | Errors: 314

[9/10] day5_consultation12
✓ WER: 14.49 %
  Ref words: 821 | Errors: 119

[10/10] day5_consultation03
✓ WER: 12.17 %
  Ref words: 2522 | Errors: 307

WHISPER LARGE-V3 PILOT WER SUMMARY
Consultations evaluated: 10
Total reference words: 16598
Total hypothesis words: 15728

Corpus WER: 16.31 %
Macro-average WER: 16.28 %
Median WER: 15.74 %

Error totals:
Substit